In [1]:
print("hello")

hello


In [2]:
pip install sentence-transformers faiss-cpu pypdf ollama langchain-text-splitters


In [3]:

from pypdf import PdfReader

def load_pdf(pdf_path):

    reader = PdfReader(pdf_path)

    text = ""

    for page in reader.pages:

        text += page.extract_text() + "\n"

    return text


In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def create_chunks(text):

    splitter = RecursiveCharacterTextSplitter(

        chunk_size=500,

        chunk_overlap=100

    )

    return splitter.split_text(text)



In [7]:

from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(

    "all-MiniLM-L6-v2"

)

embeddings = embedding_model.encode(

    chunks,

    show_progress_bar=True

)



Batches: 100%|██████████| 1/1 [00:00<00:00,  4.89it/s]


In [6]:
import faiss
import numpy as np

text = load_pdf("C:/Users/Harini/Downloads/HARINI N Internship Certificate.pdf")

chunks = create_chunks(text)

embeddings = embedding_model.encode(chunks)

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings))


In [9]:
def retrieve_context(query,k=3):

    query_embedding = embedding_model.encode([query])

    distances,indices = index.search(

        np.array(query_embedding),

        k

    )

    results = []

    for idx in indices[0]:

        results.append(chunks[idx])

    return results




In [10]:
import ollama

def ask_llm(question,context):

    prompt = f"""

You are a document assistant.

Answer ONLY using the context.

Context:

{context}

Question:

{question}

"""

    response = ollama.chat(

        model='gemma3:4b',

        messages=[

            {"role":"user","content":prompt}

        ]

    )

    return response['message']['content']


In [11]:
def rag_agent(question):

    contexts = retrieve_context(question)

    combined_context = "\n\n".join(contexts)

    answer = ask_llm(

        question,

        combined_context

    )

    return answer

In [12]:
question = input("Ask Question: ")

print(rag_agent(question))



Data Science & Analytics
